In [2]:
import csv
import torch
from sentence_transformers import SentenceTransformer
import sys
sys.path.append("/workspaces/hallucilation_in_llm")
from models.hf_model import HFModel

from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics

# ===========================
# TÜRKÇE ÖRNEK PROMPTLAR VE GROUND TRUTH
# ===========================
prompts_tr = [
    "Fransa'nın başkenti neresidir?",
    "Fotosentezi açıklayın.",
    "Mona Lisa'yı kim yaptı?",
    "Kara delikleri tanımlayın.",
    "Kuantum dolanıklığı nedir?"
]

ground_truth_tr = [
    "Paris",
    "Fotosentez, bitkilerin ışığı kimyasal enerjiye dönüştürdüğü süreçtir.",
    "Leonardo da Vinci",
    "Kara delik, hiçbir şeyin kaçamadığı kadar güçlü yerçekimine sahip uzay bölgesidir.",
    "Kuantum dolanıklığı, parçacıkların mesafeye bakılmaksızın birbirleriyle ilişkili olduğu bir fenomendir."
]

# ===========================
# MODEL VE PIPELINE HAZIRLA
# ===========================
model = HFModel("gpt2")  # Örnek: yerel GPT2

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="tr")
}

final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)

# ===========================
# PIPELINE'İ ÇALIŞTIR
# ===========================
results = []

for prompt in prompts_tr:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)

# ===========================
# GROUND TRUTH SEMANTIC SIMILARITY
# ===========================
class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r['responses'] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_tr)
similarity_scores = gt_eval.compute_similarity(responses_list)

# ===========================
# STATISTICS VE CALIBRATION
# ===========================
final_scores = [r['evaluation']['final_score'] for r in results]

# Pearson ve Spearman
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

# AUROC ve PR-AUC (StatisticalAnalyzer sınıfı içinde binary dönüşüm yapıyor)
auroc = StatisticalAnalyzer.auroc(final_scores, similarity_scores, threshold=0.5)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, similarity_scores, threshold=0.5)

# Brier ve ECE
brier = CalibrationMetrics.brier_score(final_scores, similarity_scores)
ece = CalibrationMetrics.expected_calibration_error(final_scores, similarity_scores)

# ===========================
# SONUÇLARI CSV'YE KAYDET
# ===========================
csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox_Uncertainty",
    "Graybox_Uncertainty",
    "Whitebox_Uncertainty",
    "Semantic_Uncertainty",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Pearson_Corr",
    "Spearman_Corr",
    "AUROC",
    "PR_AUC",
    "Brier_Score",
    "ECE"
]

csv_file = "pipeline_results_tr.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_tr[i],
            "Responses": "; ".join(r['responses']),
            "Blackbox_Uncertainty": r['uncertainty'].get('blackbox', None),
            "Graybox_Uncertainty": r['uncertainty'].get('graybox', None),
            "Whitebox_Uncertainty": r['uncertainty'].get('whitebox', None),
            "Semantic_Uncertainty": r['uncertainty'].get('semantic', None),
            "Final_Score": r['evaluation']['final_score'],
            "Decision": r['decision'],
            "Semantic_Similarity": similarity_scores[i],
            "Pearson_Corr": pearson,
            "Spearman_Corr": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier_Score": brier,
            "ECE": ece
        })

print(f"Tüm sonuçlar '{csv_file}' dosyasına kaydedildi.")


ImportError: cannot import name 'HFModel' from 'models.hf_model' (/workspaces/hallucilation_in_llm/models/hf_model.py)